In [1]:
import os
import fitz  # PyMuPDF
import cv2
import numpy as np

RAW = "../data/raw/"
PROC = "../data/processed/"
os.makedirs(PROC, exist_ok=True)

def pdf_to_images(pdf_path, out_dir, dpi=300):
    doc = fitz.open(pdf_path)
    base = os.path.splitext(os.path.basename(pdf_path))[0]
    paths = []
    for i, page in enumerate(doc):
        mat = fitz.Matrix(dpi/72, dpi/72)
        pix = page.get_pixmap(matrix=mat, alpha=False)
        out_path = os.path.join(out_dir, f"{base}_page_{i+1}.png")
        pix.save(out_path)
        paths.append(out_path)
    return paths

def deskew(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    inv = cv2.bitwise_not(gray)
    thr = cv2.threshold(inv, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)[1]
    coords = np.column_stack(np.where(thr > 0))
    angle = 0.0
    if coords.size > 0:
        angle = cv2.minAreaRect(coords)[-1]
        angle = -(90 + angle) if angle < -45 else -angle
    (h, w) = image.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    return cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

def preprocess_image(in_path, out_path):
    img = cv2.imread(in_path)
    if img is None:
        return
    img = deskew(img)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (3,3), 0)
    thr = cv2.adaptiveThreshold(blur,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                cv2.THRESH_BINARY, 31, 12)
    cv2.imwrite(out_path, thr)

# Convert all PDFs + clean any images in data/raw
for fn in os.listdir(RAW):
    fp = os.path.join(RAW, fn)
    if fn.lower().endswith(".pdf"):
        pages = pdf_to_images(fp, PROC, dpi=300)
        for p in pages:
            preprocess_image(p, p)
            print("[PROC] PDF page:", p)
    elif fn.lower().endswith((".png",".jpg",".jpeg")):
        out = os.path.join(PROC, fn)
        preprocess_image(fp, out)
        print("[PROC] Image:", out)

print("✅ Preprocessing done. Check folder:", PROC)


[PROC] PDF page: ../data/processed/2023_August_Statement (6)_page_1.png
[PROC] PDF page: ../data/processed/2023_August_Statement (6)_page_2.png
[PROC] PDF page: ../data/processed/2023_August_Statement (6)_page_3.png
[PROC] PDF page: ../data/processed/2023_August_Statement (6)_page_4.png
[PROC] PDF page: ../data/processed/2023_August_Statement (6)_page_5.png
[PROC] PDF page: ../data/processed/2023_December_Statement (4)_page_1.png
[PROC] PDF page: ../data/processed/2023_December_Statement (4)_page_2.png
[PROC] PDF page: ../data/processed/2023_December_Statement (4)_page_3.png
[PROC] PDF page: ../data/processed/2023_December_Statement (4)_page_4.png
[PROC] PDF page: ../data/processed/2023_July_Statement (7)_page_1.png
[PROC] PDF page: ../data/processed/2023_July_Statement (7)_page_2.png
[PROC] PDF page: ../data/processed/2023_July_Statement (7)_page_3.png
[PROC] PDF page: ../data/processed/2023_July_Statement (7)_page_4.png
[PROC] PDF page: ../data/processed/2023_October_Statement (5)_pa